#### Homework: 1、使用data/zhouyi_dataset_handmade.csv数据训练chatGLM3 2、epoch=50 handmade_dataset overfit

In [1]:
import torch
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

# 模型ID或本地路径
model_name_or_path = 'THUDM/chatglm3-6b'

In [2]:
_compute_dtype_map = {
    'fp32': torch.float32,
    'fp16': torch.float16,
    'bf16': torch.bfloat16
}

# QLoRA 量化配置
q_config = BitsAndBytesConfig(load_in_4bit=True,
                              bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=_compute_dtype_map['bf16'])

# 加载量化后模型(与微调的 revision 保持一致）
base_model = AutoModel.from_pretrained(model_name_or_path,
                                      quantization_config=q_config,
                                      device_map='auto',
                                      trust_remote_code=True,
                                      revision='b098244')

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [4]:
base_model.requires_grad_(False)
base_model.eval()

ChatGLMForConditionalGeneration(
  (transformer): ChatGLMModel(
    (embedding): Embedding(
      (word_embeddings): Embedding(65024, 4096)
    )
    (rotary_pos_emb): RotaryEmbedding()
    (encoder): GLMTransformer(
      (layers): ModuleList(
        (0-27): 28 x GLMBlock(
          (input_layernorm): RMSNorm()
          (self_attention): SelfAttention(
            (query_key_value): Linear4bit(in_features=4096, out_features=4608, bias=True)
            (core_attention): CoreAttention(
              (attention_dropout): Dropout(p=0.0, inplace=False)
            )
            (dense): Linear4bit(in_features=4096, out_features=4096, bias=False)
          )
          (post_attention_layernorm): RMSNorm()
          (mlp): MLP(
            (dense_h_to_4h): Linear4bit(in_features=4096, out_features=27392, bias=False)
            (dense_4h_to_h): Linear4bit(in_features=13696, out_features=4096, bias=False)
          )
        )
      )
      (final_layernorm): RMSNorm()
    )
    (output_la

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path,
                                          trust_remote_code=True,
                                          revision='b098244')

## 使用原始 ChatGLM3-6B 模型

In [6]:
input_text = "解释下坤卦是什么？"

In [7]:
response, history = base_model.chat(tokenizer, query=input_text)

In [8]:
print(response)

坤卦是《易经》中的一个卦象，它是由两个阴爻夹一个阳爻构成，象征着地。坤卦的卦辞描述了地的特性，如承载、包容、滋养等。坤卦在八宫图（后天八卦图）中代表西南方位，与事业、努力、顺从、阴柔、承载等有关。

在《易经》的六十四卦中，坤卦是第二个卦。它反映了大地承载万物的特性，象征着人类要效仿地的承载和包容的品质，以求达到和谐共生的境地。因此，坤卦教育人们要学会关爱他人，努力付出，以实现共同发展。


#### 询问一个64卦相关问题（应该不在 ChatGLM3-6B 预训练数据中）

In [9]:
response, history = base_model.chat(tokenizer, query="周易中的讼卦是什么？", history=history)
print(response)

讼卦是《易经》中的一个卦象，它由三个阴爻夹一个阳爻构成，象征着天。讼卦代表了诉讼、争端、矛盾等纷争现象。在八宫图（后天八卦图）中，讼卦位于东南方位，与法律、争斗、困扰、矛盾等有关。

讼卦的卦辞描述了争端的特点，如坚持己见、不听他人意见、互相指责等。它提醒人们在处理争端时要保持冷静、客观，要倾听双方的意见，寻求和解，避免矛盾升级。

在《易经》的六十四卦中，讼卦是第三十个卦。它教育人们要学会处理矛盾和争端，以达到和谐共处的境地。同时，讼卦也提醒人们要遵守法律法规，遵循社会秩序，以维护社会的和谐稳定。


## 使用微调后的 ChatGLM3-6B

### 加载 QLoRA Adapter(Epoch=50, handmade-dataset) - 请根据训练时间戳修改 timestamp 

In [10]:
from peft import PeftModel, PeftConfig

epochs = 50
# timestamp = "20240118_164514"
timestamp = "20240314_151150"

peft_model_path = f"models/{model_name_or_path}-epoch{epochs}-{timestamp}"

config = PeftConfig.from_pretrained(peft_model_path)
qlora_model = PeftModel.from_pretrained(base_model, peft_model_path)
training_tag=f"ChatGLM3-6B(Epoch=50, handmade-dataset)-{timestamp}"

In [11]:
def compare_chatglm_results(query, base_model, qlora_model, training_tag):
    base_response, base_history = base_model.chat(tokenizer, query)

    inputs = tokenizer(query, return_tensors="pt").to(0)
    ft_out = qlora_model.generate(**inputs, max_new_tokens=512)
    ft_response = tokenizer.decode(ft_out[0], skip_special_tokens=True)
    
    print(f"问题：{query}\n\n原始输出：\n{base_response}\n\n\n微调后（{training_tag}）：\n{ft_response}")
    return base_response, ft_response

### 微调前后效果对比

In [12]:
base_response, ft_response = compare_chatglm_results("解释下坤卦是什么？", base_model, qlora_model, training_tag)

问题：解释下坤卦是什么？

原始输出：
坤卦是《易经》中的卦象之一，由两个阴爻夹一个阳爻构成，象征着地，表示大地。坤卦象征着大地，它“载物的load”，承受着世界的多样性，并以其广阔的胸怀容纳万物。坤卦的卦辞是“元、亨、利、贞”，表示大吉大利，吉利的贞卜。

坤卦的卦象是六爻中阴爻居中，阳爻在顶端，象征大地承载阳光，阳光赖大地才能照耀世界。这表示了陰柔包容阳刚，事物的发展依赖于相对矛盾的双方，彼此制约和融合，才能达到和谐的状态。

坤卦的卦辞：

坤：元，亨，利，贞。

西南行，吉，利。


微调后（ChatGLM3-6B(Epoch=50, handmade-dataset)-20240314_151150）：
[gMASK]sop 解释下坤卦是什么？ 坤卦是《易经》中的卦象之一，也是六十四卦中的第二个卦。坤卦是由两个阴爻夹一个阳爻构成，象征着地，表示大地。坤卦的卦象为二阴一阳，象征着大地承载万物，阴柔顺从，含蓄内敛。它代表着顺从、承载、包容、稳重、谨慎等性格特点，也象征着大地上的万物，是万物的承载者。在《易经》中，坤卦与乾卦相互配合，象征着天与地，是宇宙的基本构成部分。坤卦的阳爻象征着万物，阳爻在阴爻之中，表示万物生长，顺从地性，是天地的补充。通过坤卦和乾卦的相互配合，它们共同构成了宇宙的基本模型，是宇宙的基本原则。


In [13]:
base_response, ft_response = compare_chatglm_results("周易中的讼卦是什么", base_model, qlora_model, training_tag)

问题：周易中的讼卦是什么

原始输出：
周易中的讼卦是卦象为雷泽象，卦名来自于古人诉讼的诉讼。讼卦象征诉讼，表明 Conflict（冲突）和 Dispute（争议）的存在，同时也表明通过诉讼可以解决争议，恢复和平。

讼卦的卦象是上卦为泽，下卦为雷，雷从下卦升起， into上卦，表示通过诉讼解决争议，恢复和平。

讼卦的卦象代表了一种困境，需要通过法律和公正来解决争议。由于雷泽相交，表示公正和公平，但也表示冲突和争议。因此，讼卦象征着在公正和公平的过程中解决冲突和争议。

总结起来，讼卦表明在处理争议和冲突时需要公正、公平和理性的态度，同时也表明解决争议需要经过法律程序和公正的裁决。


微调后（ChatGLM3-6B(Epoch=50, handmade-dataset)-20240314_151150）：
[gMASK]sop 周易中的讼卦是什么？讼卦是周易中的第卦，卦象是上乾下坤，乾为天，坤为地。乾为阳，坤为阴，天地相交，万物相应。讼卦象征诉讼、争端、诉讼的含义，是“卦”中唯一的一个。
《周易》中的卦辞：讼。有孚，窒惕，中吉，终凶。利见大人，不利涉大川。象：天与地相交，卦风:下乾上坤。象辞：天（乾）德（德行）无穷，无所不包，穷高广深；地（坤）德厚，永居中央，来顺承天。得此卦者，初交涉者；初脱涉者；阴乘阳，凶；阴顺阳，吉。  translator


In [14]:
base_response, ft_response = compare_chatglm_results("师卦是什么？", base_model, qlora_model, training_tag)

问题：师卦是什么？

原始输出：
师卦，即师卦（下离上坤）为火，卦形为 three lines and one line。师卦是由坤卦下艮卦上构成，上坤为火，下艮为火，火生火，火克土。师卦象征军队，表示军队的盛衰，兵力的强弱，以及军队的统治和领导。师卦时空观念：时运：军运：战局：兵种：兵法。得此卦者，军队强大，兵强马壮，将军得力，宜决策敢战，胜算在握。

师卦原文：师。贞，如心。如心，坎。火克土。坎上坤下。火生火，火克土。坎：初坤上艮。火生火，火克土。师卦。白话文解释：师卦是由坤卦下艮卦上构成的。坤卦和艮卦都是火，火生火，火克土。坤卦下面是艮卦，火生火，火克土。师卦 tell you that the military is powerful and strong, the army is powerful and strong, the general is well-respected, and the battle is won.

师卦的卦象是：下坤上艮。坤卦下面是艮卦，火生火，火克土。火遇见火，火生火，火克土。

师卦的卦义：师卦纯阳，火生火，火克土，战胜之兆，适合战争，不宜 Send Offer。

师卦的吉凶：吉，宜出口，不宜岛。


微调后（ChatGLM3-6B(Epoch=50, handmade-dataset)-20240314_151150）：
[gMASK]sop 师卦是什么？师卦是易经中的卦象之一，属于师卦（师贞）大吉卦。师卦是由两个阴爻夹一个阳爻构成，如师。如：师。如：师。如：师。 师卦象辞：师，贞，吉，无咎。象：地中有水，师。君子以容民畜众。白话文解释：师卦象征军队指挥，无灾祸。卦象：下卦为坎（水），上卦为坤（地），如师坎之合。君子应容纳众人，宜团结协作。得此卦者，宜 training、教育、传播知识，以培养人才。得此卦者，不宜从事战争刚结束的战争贸易，而宜在和平时期贸易。得此卦者，宜接受他人的帮助，以克服困难。白话文解释：师卦象征军队指挥，无灾祸。卦象：下坎上坤，师卦合象。君子应容纳众人，宜团结协作。得此卦者，宜培养人才。得此卦者，不宜从事战争贸易，而宜在和平时期贸易。得此卦者，宜接受他人的帮助，以克服困难。传统文化中，师卦是“师”字卦，表示军队指挥，无灾祸，宜培养人才。遇此卦，宜接受他人的帮助，以克服困难。
